# How to use
- เปิด Playground mode ด้วยการกด `File` -> `Open in playground mode`
- copy url ของ Google Sheet ที่จะใช้เป็น sheet สำหรับ validate ข้อมูลงบมาใส่ในช่องข้างล่างนี้
- กด Run all
- ในช่อง "อัปโหลดไฟล์" จะมีตัวเลือกให้กด `Choose Files`
- ให้เลือกไฟล์ csv ของ budget tree ทั้งหมดที่สร้างมาจาก [wevis-openbudget-ocr](https://github.com/wevisdemo/wevis-openbudget-ocr)

In [72]:
#@title url ของ Google Sheet
SHEET_URL = 'https://docs.google.com/spreadsheets/d/1qkfZ-Mt4XycAuK46X4HakdFAj-sboBiL9Zf8r1q2Zdk/edit?gid=0#gid=0' # @param {"type": "string"}

In [73]:
#@title อัปโหลดไฟล์
#@markdown Run cell นี้และกด `Choose Files` เพื่ออัปโหลดไฟล์ csv ที่สร้างมาจาก wevis-openbudget-ocr โดยให้ชื่อไฟล์ตรงกับชื่อกระทรวง

from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "Uploading_Data_Colab_1.xlsx" with length 9000 bytes'.format(
      name=fn, length=len(uploaded[fn])))

# Ignore This

In [74]:
# Connect to Google Account & Google Sheet
from google.colab import auth
import gspread
from google.auth import default

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

In [75]:
sheet = gc.open_by_url(SHEET_URL)

In [76]:
sheet.worksheets

<bound method Spreadsheet.worksheets of <Spreadsheet '[OpenBudget] test validation setup' id:1qkfZ-Mt4XycAuK46X4HakdFAj-sboBiL9Zf8r1q2Zdk>>

## Curate CSV Files

In [77]:
!pip install gspread-formatting -q

In [78]:
BASE_URL = "https://bbstore.bb.go.th/cms/"
def add_page_url(df: pd.DataFrame, url: str):
  df['page_url'] = df.apply(
      lambda row: f'=HYPERLINK("{BASE_URL}{row["document"]}?#page={row["page"]}", "page {row["page"]}")',
      axis=1
  )

  # Rearrange columns
  columns = df.columns.tolist()
  columns.insert(-2, columns.pop(columns.index('page_url')))
  df = df.reindex(columns=columns)
  return df

In [79]:
from gspread_formatting import *

def format_sheet(worksheet, df: pd.DataFrame):
  batch = batch_updater(sheet)
  sheet_rows = len(df.index) + 1

  # Header
  header_fmt = CellFormat(
    backgroundColor=Color(0.7, 0.7, 0.7),
    textFormat=TextFormat(bold=True),
  )
  batch.format_cell_range(worksheet, 'A1:S1', header_fmt)

  # Amount & Page URL column
  amount_col_fmt = CellFormat(
    backgroundColor=Color(0.8, 1, 0.9),
    textFormat=TextFormat(bold=True),
  )
  page_col_fmt = CellFormat(
    backgroundColor=Color(0.91, 0.94, 1),
  )
  batch.format_cell_range(worksheet, f'N2:N{sheet_rows}', amount_col_fmt)
  batch.format_cell_range(worksheet, f'Q2:Q{sheet_rows}', page_col_fmt)

  # Add condition formatting
  # Error
  rules = get_conditional_format_rules(worksheet)
  rules.clear()
  for formula, color in [
    ('=$A2<>""', Color(1, 0.4, 0.4)),
    ('=$B2="BUDGETARY_UNIT"', Color(0.85, 0.9, 1)),
    ('=$B2="BUDGET_PLAN"', Color(0.85, 0.85, 0.85)),
    ('=OR($B2="OUTPUT", $B2="PROJECT")', Color(0.95, 0.95, 0.95)),
  ]:
    rule = ConditionalFormatRule(
        ranges=[GridRange.from_a1_range(f'A2:S{sheet_rows}', worksheet)],
        booleanRule=BooleanRule(
            condition=BooleanCondition('CUSTOM_FORMULA', [formula]),
            format=CellFormat(backgroundColor=color)
        )
    )
    rules.append(rule)
  rules.save()

  batch.execute()
  worksheet.hide_columns(14, 16)

In [80]:
import os

files = os.listdir()
files = [
    f for f in files if f.endswith('.csv')
]
files

['งบกลาง.csv', 'ส่วนราชการในพระองค์.csv', 'องค์กรปกครองส่วนท้องถิ่น.csv']

In [81]:
import re
import pandas as pd

sheet_titles = [
    re.sub(r"[^\u0e00-\u0e59]", "", sh.title) for sh in sheet.worksheets()
]

for ministry_csv in files:
  ministry_name = re.sub(r"(\(.*\))?\..*", "", ministry_csv).strip()
  if ministry_name in sheet_titles:
    print(f'Skip {ministry_name}')
    continue

  # Load data
  df = pd.read_csv(ministry_csv)
  df = add_page_url(df, BASE_URL)
  df = df.fillna('')
  # Add new sheet
  new_sheet = sheet.add_worksheet(title=ministry_name, rows=len(df.index), cols=len(df.columns))
  # Update Sheet
  new_sheet.update(
      [df.columns.values.tolist()] + df.values.tolist(),
      value_input_option='USER_ENTERED'
  )
  print(f'Add {ministry_name}')

  # Add format
  format_sheet(new_sheet, df)

Add งบกลาง
Add ส่วนราชการในพระองค์
Add องค์กรปกครองส่วนท้องถิ่น
